### 0. Environment Setup
Install the necessary libraries for the transformer, datasets, and N-gram models.

In [1]:
!pip install transformers datasets torch nltk pandas tqdm

### 1. NLTK Setup & N-Gram Baseline Model
Downloads the Brown corpus, trains a Trigram model with Laplace smoothing, and suppresses Hugging Face Hub symlink warnings.

In [2]:
import os
import re
import math
import nltk
from dataclasses import dataclass
from typing import List, Tuple, Optional
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.notebook import tqdm
from transformers import AutoModelForMaskedLM, AutoTokenizer

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('brown', quiet=True)

from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import Laplace
from nltk.tokenize import word_tokenize
from nltk.corpus import brown

print("Training N-Gram Baseline Model on Brown Corpus...")
n_gram_size = 3
train_data, padded_sents = padded_everygram_pipeline(n_gram_size, brown.sents())
ngram_model = Laplace(n_gram_size)
ngram_model.fit(train_data, padded_sents)
print("N-Gram Model trained successfully.\n")

class NGramScorer:
    def __init__(self, model, n):
        self.model = model
        self.n = n
        
    def sentence_log_score(self, sentence: str) -> float:
        tokens = word_tokenize(sentence.lower())
        test_data, _ = padded_everygram_pipeline(self.n, [tokens])
        
        score = 0.0
        for ngram in test_data:
            for item in ngram:
                score += self.model.logscore(item[-1], item[:-1])
        return score

    def compare_candidates(self, candidate1: str, candidate2: str) -> Tuple[float, float, str]:
        score1 = self.sentence_log_score(candidate1)
        score2 = self.sentence_log_score(candidate2)
        predicted = "1" if score1 >= score2 else "2"
        return score1, score2, predicted

Training N-Gram Baseline Model on Brown Corpus...
N-Gram Model trained successfully.



### 2. Transformer (BERT) Setup
Initializes the pre-trained Masked Language Model and defines the pseudo-log-likelihood (PLL) scorer.

In [3]:
class MaskedLMPLLScorer:
    def __init__(self, model_name: str, device: torch.device):
        self.model_name = model_name
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
        self.model.eval()

    @torch.no_grad()
    def sentence_pll(self, sentence: str) -> float:
        enc = self.tokenizer(sentence, return_tensors="pt")
        input_ids = enc["input_ids"].to(self.device)
        attention_mask = enc["attention_mask"].to(self.device)

        seq_len = input_ids.size(1)
        pll = 0.0
        special_mask = self.tokenizer.get_special_tokens_mask(
            input_ids[0].tolist(), already_has_special_tokens=True
        )

        for pos in range(seq_len):
            if special_mask[pos] == 1:
                continue

            original_token_id = input_ids[0, pos].item()
            masked_ids = input_ids.clone()
            masked_ids[0, pos] = self.tokenizer.mask_token_id

            outputs = self.model(input_ids=masked_ids, attention_mask=attention_mask)
            logits = outputs.logits[0, pos]
            log_probs = torch.log_softmax(logits, dim=-1)
            pll += float(log_probs[original_token_id].item())

        return pll

    def compare_candidates(self, candidate1: str, candidate2: str) -> Tuple[float, float, str]:
        score1 = self.sentence_pll(candidate1)
        score2 = self.sentence_pll(candidate2)
        predicted = "1" if score1 >= score2 else "2"
        return score1, score2, predicted

### 3. Data Structures & Loading Utilities
Defines the dataset structures and the logic to sequentially sample WinoGrande and replace the blanks.

In [4]:
@dataclass
class Example:
    sentence: str
    option1: str
    option2: str
    answer: str
    example_id: str

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def replace_blank(sentence: str, replacement: str) -> str:
    if "_" not in sentence:
        raise ValueError(f"No underscore placeholder found in sentence: {sentence}")
    return normalize_space(sentence.replace("_", replacement, 1))

def load_winogrande_examples(split: str, max_items: int) -> List[Example]:
    ds = load_dataset("winogrande", "winogrande_xl", split=split)
    rows = list(ds)
    
    if max_items < len(rows):
        rows = rows[:max_items]

    examples = []
    for idx, row in enumerate(rows):
        row_id = str(row.get("qID", row.get("id", idx)))
        
        examples.append(Example(
            sentence=row["sentence"],
            option1=row["option1"],
            option2=row["option2"],
            answer=str(row["answer"]),
            example_id=row_id,
        ))
    return examples

### 4. Main Evaluation Loop
Runs both models over the WinoGrande subset sequentially, calculates accuracy, and outputs a comparative DataFrame.

In [5]:
def run_baseline_comparison():
    MODEL_NAME = "bert-base-uncased"
    MAX_ITEMS = 300 
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    ngram_scorer = NGramScorer(ngram_model, n_gram_size)
    bert_scorer = MaskedLMPLLScorer(model_name=MODEL_NAME, device=device)
    
    examples = load_winogrande_examples("validation", MAX_ITEMS)
    print(f"Loaded the first {len(examples)} examples from WinoGrande.\n")
    
    results = []
    
    for ex in tqdm(examples, desc="Evaluating Models"):
        cand1 = replace_blank(ex.sentence, ex.option1)
        cand2 = replace_blank(ex.sentence, ex.option2)
        
        ng_s1, ng_s2, ng_pred = ngram_scorer.compare_candidates(cand1, cand2)
        ng_correct = (ng_pred == ex.answer)
        
        bert_s1, bert_s2, bert_pred = bert_scorer.compare_candidates(cand1, cand2)
        bert_correct = (bert_pred == ex.answer)
        
        results.append({
            "example_id": ex.example_id,
            "sentence_template": ex.sentence,
            "gold_answer": ex.answer,
            "ngram_prediction": ng_pred,
            "ngram_correct": ng_correct,
            "bert_prediction": bert_pred,
            "bert_correct": bert_correct
        })
        
    df_results = pd.DataFrame(results)
    
    ngram_acc = df_results["ngram_correct"].mean() * 100
    bert_acc = df_results["bert_correct"].mean() * 100
    
    print("\n--- BASELINE RESULTS ---")
    print(f"N-Gram Model Accuracy: {ngram_acc:.2f}%")
    print(f"BERT Model Accuracy:   {bert_acc:.2f}%")
    
    return df_results

results_df = run_baseline_comparison()
results_df.head()

Using device: cpu


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded the first 300 examples from WinoGrande.



Evaluating Models:   0%|          | 0/300 [00:00<?, ?it/s]


--- BASELINE RESULTS ---
N-Gram Model Accuracy: 48.67%
BERT Model Accuracy:   50.00%


,example_id,sentence_template,gold_answer,ngram_prediction,ngram_correct,bert_prediction,bert_correct
0,0,Sarah was a much better surgeon than Maria so ...,2,2,True,2,True
1,1,Sarah was a much better surgeon than Maria so ...,1,2,False,2,False
2,2,They were worried the wine would ruin the bed ...,2,2,True,2,True
3,3,Terry tried to bake the eggplant in the toaste...,1,1,True,1,True
4,4,"At night, Jeffrey always stays up later than H...",1,2,False,1,True
